# 03 - Silver
## 06 - Silver Reference Data

### Goal
- clean all bronze reference tables into stable silver lookup tables
- standardize identifiers and region values
- handle duplicates
- produce reliable join keys for silver integrated and gold layers

In [0]:
%run ../01_setup/00_config

In [0]:
from pyspark.sql import functions as F

## Asset reference

Standardize `asset_id`, `region`, and `asset_type`. Remove `_rescued_data` and `ingestion_ts`. Deduplicate on `asset_id`.

In [0]:
asset_df = spark.table(bronze_asset_reference_table)

silver_asset_df = (
    asset_df
    .filter(F.col("asset_id").isNotNull() & (F.trim(F.col("asset_id")) != ""))
    .filter(F.col("region").isNotNull() & (F.trim(F.col("region")) != ""))
    .withColumn("asset_id",   F.upper(F.trim(F.col("asset_id"))))
    .withColumn("asset_name", F.initcap(F.trim(F.col("asset_name"))))
    .withColumn("region",     F.upper(F.trim(F.col("region"))))
    .withColumn("asset_type", F.upper(F.trim(F.col("asset_type"))))
    .dropDuplicates(["asset_id"])
    .drop("_rescued_data", "ingestion_ts")
)

display(silver_asset_df)
print(f"Bronze rows: {asset_df.count()} - Silver rows: {silver_asset_df.count()}")

## Region reference

Standardize `region` and `country_code`. Deduplicate on `region`.

In [0]:
region_df = spark.table(bronze_region_reference_table)

silver_region_df = (
    region_df
    .filter(F.col("region").isNotNull() & (F.trim(F.col("region")) != ""))
    .withColumn("region",         F.upper(F.trim(F.col("region"))))
    .withColumn("country_code",   F.upper(F.trim(F.col("country_code"))))
    .withColumn("operating_zone", F.upper(F.trim(F.col("operating_zone"))))
    .dropDuplicates(["region"])
    .drop("ingestion_ts")
)

display(silver_region_df)
print(f"Bronze rows: {region_df.count()} - Silver rows: {silver_region_df.count()}")

## Market reference

Standardize `market_type`. Deduplicate on `market_type`.

In [0]:
market_df = spark.table(bronze_market_reference_table)

silver_market_df = (
    market_df
    .filter(F.col("market_type").isNotNull() & (F.trim(F.col("market_type")) != ""))
    .withColumn("market_type",  F.upper(F.trim(F.col("market_type"))))
    .withColumn("description",  F.initcap(F.trim(F.col("description"))))
    .dropDuplicates(["market_type"])
    .drop("ingestion_ts")
)

display(silver_market_df)
print(f"Bronze rows: {market_df.count()} - Silver rows: {silver_market_df.count()}")

## Event type reference

Standardize `event_type` and `category`. Deduplicate on `event_type`.

In [0]:
event_type_df = spark.table(bronze_event_type_reference_table)

silver_event_type_df = (
    event_type_df
    .filter(F.col("event_type").isNotNull() & (F.trim(F.col("event_type")) != ""))
    .withColumn("event_type", F.upper(F.trim(F.col("event_type"))))
    .withColumn("category",   F.upper(F.trim(F.col("category"))))
    .dropDuplicates(["event_type"])
    .drop("ingestion_ts")
)

display(silver_event_type_df)
print(f"Bronze rows: {event_type_df.count()} - Silver rows: {silver_event_type_df.count()}")

## Weather alert reference

Standardize `weather_alert_level`. Deduplicate on `weather_alert_level`.

In [0]:
weather_alert_df = spark.table(bronze_weather_alert_reference_table)

silver_weather_alert_df = (
    weather_alert_df
    .filter(F.col("weather_alert_level").isNotNull() & (F.trim(F.col("weather_alert_level")) != ""))
    .withColumn("weather_alert_level", F.upper(F.trim(F.col("weather_alert_level"))))
    .dropDuplicates(["weather_alert_level"])
    .drop("ingestion_ts")
)

display(silver_weather_alert_df)
print(f"Bronze rows: {weather_alert_df.count()} - Silver rows: {silver_weather_alert_df.count()}")

## Save all silver reference tables

In [0]:
silver_asset_df.write.mode("overwrite").saveAsTable(silver_asset_reference_table)
silver_region_df.write.mode("overwrite").saveAsTable(silver_region_reference_table)
silver_market_df.write.mode("overwrite").saveAsTable(silver_market_reference_table)
silver_event_type_df.write.mode("overwrite").saveAsTable(silver_event_type_reference_table)
silver_weather_alert_df.write.mode("overwrite").saveAsTable(silver_weather_alert_reference_table)

print(f"Saved: {silver_asset_reference_table}")
print(f"Saved: {silver_region_reference_table}")
print(f"Saved: {silver_market_reference_table}")
print(f"Saved: {silver_event_type_reference_table}")
print(f"Saved: {silver_weather_alert_reference_table}")